# Подробнее про функции

## Функция — это объект

В Python функция — это **полноценный объект** класса `function`.
Это означает, что с ней можно делать всё то же, что и с любым другим объектом:

- присваивать переменной
- передавать как аргумент
- возвращать из другой функции
- хранить в списке или словаре


In [6]:
def greet(name):
    """Greeting user"""
    return f"Hello, {name}!"

print(type(greet))
print(type(lambda x: x))  # lambda тоже function
print(type(greet) is type(lambda x: x))

<class 'function'>
<class 'function'>
True


In [7]:
# Функция — это просто объект, которому можно дать другое имя
say_hello = greet
print(say_hello("Alice"))  # Hello, Alice!

# Можно хранить в списке
funcs = [greet, str.upper, len]
print([f.__name__ for f in funcs])

Hello, Alice!
['greet', 'upper', 'len']


## Иерархия типов: родители `function`

Раз функция — это объект, у её класса есть родители. Давай разберёмся, где `function` живёт в иерархии типов Python.

- Класс `function` напрямую наследуется от `object` — никаких промежуточных классов
- `types.FunctionType` — это просто удобный псевдоним, не отдельный класс

In [13]:
import types

def greet(name):
    return f"Hello, {name}!"

# Проверяем тип
print(type(greet))                        # <class 'function'>
print(type(greet) is types.FunctionType)  # True

# MRO класса function
print(type(greet).__mro__)


<class 'function'>
True
(<class 'function'>, <class 'object'>)


## Атрибуты функции

У каждой функции есть набор встроенных атрибутов. Разберём самые важные.

In [17]:
def greet(name: str, greeting: str = "Hello") -> str:
    """Greets a user with a given greeting."""
    return f"{greeting}, {name}!"

# Основные атрибуты
print(greet.__name__)
print(greet.__doc__)
print(greet.__module__)
print(greet.__annotations__)
print(greet.__defaults__)


greet
Greets a user with a given greeting.
__main__
{'name': <class 'str'>, 'greeting': <class 'str'>, 'return': <class 'str'>}
('Hello',)


### Пользовательские атрибуты

Функции можно назначать произвольные атрибуты — они хранятся в `__dict__`.
Это редкий паттерн, но иногда полезен, например, для простого кэширования или счётчиков без использования классов.

In [37]:
def greet(name: str, greeting: str = "Hello") -> str:
    greet.call_count += 1
    return f"{greeting}, {name}!"

greet.call_count = 0  # Это атрибут конкретного объекта greet, а не класса <function>

In [ ]:
greet("Alice")
print(greet.call_count)
print(greet.__dict__)

5
{'call_count': 5}


## Функция vs callable-объект

Любой объект, у которого есть метод `__call__`, можно вызвать как функцию.
Чем тогда обычная функция отличается от такого объекта?


In [38]:
# Callable-объект
class Greeter:
    def __call__(self, name: str) -> str:
        return f"Hello, {name}!"

greeter = Greeter()

# Оба вызываются одинаково
def greet(name: str) -> str:
    return f"Hello, {name}!"

print(greet("Alice"))    # Hello, Alice!
print(greeter("Alice"))  # Hello, Alice!

# Оба проходят проверку callable
print(callable(greet))    # True
print(callable(greeter))  # True

# Но это разные типы
print(type(greet))    # <class 'function'>
print(type(greeter))  # <class 'Greeter'>

Hello, Alice!
Hello, Alice!
True
True
<class 'function'>
<class '__main__.Greeter'>


### В чём разница?

- `function` — это **встроенный тип**, его вызов обрабатывается интерпретатором напрямую, без поиска `__call__`
- callable-объект при вызове заставляет Python **искать `__call__`** через MRO — это дополнительный шаг
- callable-объект **может хранить состояние** в `self` более явно и структурированно, чем функция через `__dict__`
- функция **легче и быстрее** создаётся — не нужно инстанцировать класс


## Объект кода: `__code__`

Когда Python компилирует функцию, он создаёт объект кода — `__code__`.
Это низкоуровневое представление функции: байткод и всё необходимое для его выполнения.


In [ ]:
def greet(name: str, greeting: str = "Hello") -> str:
    message = f"{greeting}, {name}!"
    return message

code = greet.__code__

print(code.co_name)      # 'greet'        — имя функции
print(code.co_argcount)  # 2              — количество аргументов
print(code.co_varnames)  # ('name', 'greeting', 'message') — все локальные переменные
print(code.co_consts)    # (None, ...)    — константы


greet
2
('name', 'greeting', 'message')
(None, ', ', '!')


### Что внутри `__code__`?

- `co_varnames` — все локальные переменные, **сначала аргументы**, потом остальные
- `co_argcount` — количество позиционных аргументов, не считая `*args` и `**kwargs`
- `co_consts` — константы, известные на этапе компиляции


In [40]:
import dis

# Смотрим байткод
dis.dis(greet)


  2           0 LOAD_FAST                1 (greeting)
              2 FORMAT_VALUE             0
              4 LOAD_CONST               1 (', ')
              6 LOAD_FAST                0 (name)
              8 FORMAT_VALUE             0
             10 LOAD_CONST               2 ('!')
             12 BUILD_STRING             4
             14 STORE_FAST               2 (message)

  3          16 LOAD_FAST                2 (message)
             18 RETURN_VALUE


### Как читать вывод `dis`?

- **Колонка 1** — номер строки в исходном коде
- **Колонка 2** — смещение инструкции в байткоде (шаг 2, каждая инструкция занимает 2 байта)
- **Колонка 3** — название инструкции
- **Колонка 4** — индекс аргумента, в скобках — его значение

### Что происходит?

Python работает как **стековая машина** — инструкции кладут значения на стек и снимают их.
`f"{greeting}, {name}!"` разбивается на части: загрузить переменную, загрузить константу, собрать строку.

## Пространства имён и LEGB

Когда Python встречает имя переменной, он ищет её по правилу **LEGB**:

- **L**ocal — локальное пространство текущей функции
- **E**nclosing — пространства объемлющих функций (снаружи внутрь)
- **G**lobal — уровень модуля
- **B**uilt-in — встроенные имена (`len`, `print`, ...)


In [41]:
x = "global"

def outer():
    x = "enclosing"
    
    def inner():
        x = "local"
        print(x)  # local
    
    inner()
    print(x)  # enclosing

outer()
print(x)  # global


local
enclosing
global


## Замыкания

Если внутренняя функция использует переменную из объемлющей, Python создаёт **замыкание** — 
внутренняя функция сохраняет ссылку на переменную даже после того, как внешняя функция завершила работу.


In [47]:
def make_counter():
    count = 0
    
    def counter():
        nonlocal count
        count += 1
        return count
    
    return counter

c = make_counter()
del make_counter

print(c())  # 1
print(c())  # 2
print(c())  # 3

# make_counter уже завершила работу, но count жив
print(c.__closure__)                    # (<cell at 0x...>,) - ссылки на переменные из замыкания
print(c.__closure__[0].cell_contents)  # 3


1
2
3
(<cell at 0x1080c18d0: int object at 0x10422c130>,)
3


### Как это устроено?

- `__closure__` — кортеж **ячеек** (`cell objects`)
- Каждая ячейка хранит ссылку на переменную из объемлющей функции
- Если бы не кортеж замыканий, `count` был бы уничтожен вместе с фреймом `make_counter`. У него занулился бы счётчик ссылок, его удалил бы garbage collector


In [54]:
# nonlocal — явно говорим Python искать переменную в enclosing, а не создавать локальную

x = 'global'

def outer():
    x = 'nearest local'
    
    def inner():
        global x
        print(x)

    def inner_2():
        nonlocal x
        print(x)
    
    inner()
    inner_2()


outer()

global
nearest local


## Фреймы и стек вызовов (то, что печатается при ошибке)

Каждый раз, когда вызывается функция, Python создаёт **фрейм** — объект, который хранит
всё необходимое для выполнения этого вызова: локальные переменные, текущую инструкцию, ссылку на объект кода.

Фреймы выстраиваются в **стек** — каждый новый вызов кладёт фрейм сверху, возврат из функции — снимает.


In [55]:
import inspect

def inner():
    frame = inspect.currentframe()
    print(frame.f_code.co_name)        # 'inner'     — имя текущей функции
    print(frame.f_locals)              # локальные переменные
    print(frame.f_back.f_code.co_name) # 'outer'     — кто нас вызвал
    print(frame.f_lineno)              # номер текущей строки

def outer():
    x = 42
    inner()

outer()


inner
{'frame': <frame at 0x1080d5f80, file '/var/folders/vp/0v99fjwx2nlcysq5ns40bc_w0000gn/T/ipykernel_16175/3605761976.py', line 6, code inner>}
outer
8


### Что хранит фрейм?

- `f_code` — объект кода (`__code__`) функции
- `f_locals` — локальные переменные на момент обращения
- `f_globals` — глобальные переменные модуля
- `f_back` — ссылка на фрейм, который вызвал текущий
- `f_lineno` — номер текущей строки


In [57]:
# Посмотрим на весь стек вызовов
def a():
    b()

def b():
    c()

def c():
    for frame_info in inspect.stack():
        print(f"{frame_info.function:10} line {frame_info.lineno}")

a()

c          line 9
b          line 6
a          line 3
<module>   line 12
run_code   line 3579
run_ast_nodes line 3519
run_cell_async line 3336
_pseudo_sync_runner line 128
_run_cell  line 3132
run_cell   line 3077
run_cell   line 663
do_execute line 464
execute_request line 834
execute_request line 372
dispatch_shell line 478
shell_main line 621
preserve_context line 71
_run       line 80
_run_once  line 1909
run_forever line 603
start      line 211
start      line 758
launch_instance line 1075
<module>   line 18
_run_code  line 86
_run_module_as_main line 196


### Traceback — это и есть стек фреймов

Когда происходит исключение, Python просто печатает цепочку `f_back` от текущего фрейма до верхнего.


In [59]:
def a():
    b()

def b():
    c()

def c():
    raise ValueError("something went wrong")

a()

ValueError: something went wrong

## `yield` и генераторы

Обычная функция при каждом вызове создаёт новый фрейм, выполняется до конца и фрейм уничтожается.
`yield` меняет это поведение — функция может **приостановиться** и **возобновиться** с того же места.


In [62]:
def regular():
    return 1

def generator():
    yield 1

print(type(regular()))   # <class 'int'>
print(type(generator())) # <class 'generator'>

# Вызов генераторной функции не выполняет код — только создаёт объект
gen = generator()
print(gen)  # <generator object generator at 0x...>
print(gen.__next__())  # Забираем то, что возвращает генератор

<class 'int'>
<class 'generator'>
<generator object generator at 0x10a0b54d0>
1


### Как Python отличает генераторную функцию от обычной?

При компиляции Python выставляет флаг `CO_GENERATOR` в объекте кода.


In [67]:
import dis

def regular():
    return 1

def generator():
    yield 1

CO_GENERATOR_FLAG = 0x20
# 0x20 в двоичном виде — это 00100000. Операция & 0x20 проверяет, выставлен ли именно этот бит. Если результат ненулевой — флаг CO_GENERATOR установлен.

print(bool(regular.__code__.co_flags & CO_GENERATOR_FLAG))    # False
print(bool(generator.__code__.co_flags & CO_GENERATOR_FLAG))  # True


False
True


### Что внутри объекта-генератора?


In [76]:
def countdown(n: int):
    while n > 0:
        yield n
        n -= 1

gen = countdown(3)

print(gen.gi_code.co_name)  # 'countdown' — объект кода функции
print(gen.gi_frame)         # фрейм существует, функция ещё не завершена
print(gen.gi_running)       # False — сейчас не выполняется

next(gen)  # делаем шаг
print(gen.gi_frame.f_locals)  # {'n': 3} — состояние сохранено

next(gen)  # делаем шаг
print(gen.gi_frame.f_locals)  # {'n': 2} — состояние изменилось

next(gen)  # делаем шаг
print(gen.gi_frame.f_locals)  # {'n': 1} — состояние изменилось

next(gen)  # делаем шаг, окончательно выходим из функции, получаем StopIteration

countdown
<frame at 0x14a76fcc0, file '/var/folders/vp/0v99fjwx2nlcysq5ns40bc_w0000gn/T/ipykernel_16175/905466897.py', line 1, code countdown>
False
{'n': 3}
{'n': 2}
{'n': 1}


StopIteration: 

### Как работает `yield` под капотом?

1. При первом `next()` создаётся фрейм и начинается выполнение
2. Когда интерпретатор доходит до `yield` — значение возвращается наружу, **фрейм замораживается**
3. При следующем `next()` фрейм **размораживается** и выполнение продолжается со следующей инструкции после `yield`
4. Когда функция заканчивается — бросается `StopIteration`, фрейм уничтожается

В отличие от обычной функции, фрейм генератора **живёт между вызовами**.


In [79]:
# Сравним байткод
dis.dis(regular)
print("-" * 50)
dis.dis(generator)

  4           0 LOAD_CONST               1 (1)
              2 RETURN_VALUE
--------------------------------------------------
              0 GEN_START                0

  7           2 LOAD_CONST               1 (1)
              4 YIELD_VALUE
              6 POP_TOP
              8 LOAD_CONST               0 (None)
             10 RETURN_VALUE


## Ленивые вычисления и генераторные выражения

Генератор не вычисляет все значения сразу — он отдаёт их **по одному, по требованию**.
Это называется **ленивыми вычислениями** (lazy evaluation).


In [81]:
# Список вычисляет всё сразу и хранит в памяти
list_comp = [x ** 2 for x in range(10)]

# Генератор ничего не вычисляет до первого next()
gen_comp = (x ** 2 for x in range(10))

print(type(list_comp))  # <class 'list'>
print(type(gen_comp))   # <class 'generator'>

import sys
print(sys.getsizeof(list_comp))  # ~184 bytes
print(sys.getsizeof(gen_comp))   #   ~  104 bytes — независимо от размера

<class 'list'>
<class 'generator'>
184
104


In [82]:
# Генераторное выражение — это синтаксический сахар над генераторной функцией
gen_comp = (x ** 2 for x in range(5))

# Эквивалентно:
def squares():
    for x in range(5):
        yield x ** 2

# Оба ведут себя одинаково
print(list(gen_comp))    # [0, 1, 4, 9, 16]
print(list(squares()))   # [0, 1, 4, 9, 16]


[0, 1, 4, 9, 16]
[0, 1, 4, 9, 16]


### Когда это важно?

Представь, что нужно обработать файл на 10 млн строк. Список загрузит всё в память, генератор — нет.


In [83]:
# Чтение файла — классический пример ленивых вычислений
def read_large_file(path: str):
    with open(path) as f:
        for line in f:
            yield line.strip()

# В каждый момент в памяти только одна строка
gen = read_large_file("large_file.txt")
